# Meilenstein 2: Data Understanding & Preparation

## 1. Projektkontext
Ziel dieses Projekts ist die Vorhersage von Platzierungen bei Radrennen der "Grand Tours" (Tour de France, Giro d'Italia, Vuelta a España) mittels Machine Learning. Nach der erfolgreichen Datenbeschaffung und Zusammenführung der Datensätze treten wir nun in die kritische Phase der Datenaufbereitung ein.

## 2. Zielsetzung dieses Notebooks
Basierend auf dem **CRISP-DM** Framework (Cross Industry Standard Process for Data Mining) verfolgen wir in diesem Abschnitt zwei Kernziele:

### A. Data Understanding (Datenverständnis)
* **Strukturprüfung:** Identifikation von Datentypen und Skalenniveaus der Features.
* **Qualitätsanalyse:** Ermittlung von fehlenden Werten (Missing Values) und Ausreißern (Outliers).
* **Explorative Analyse:** Visualisierung von Verteilungen und Korrelationen, um erste Muster zwischen Rennprofilen (z.B. Höhenmeter) und Fahrer-Biometrie (z.B. Gewicht) zu verstehen.

### B. Data Preparation (Datenvorbereitung)
* **Cleaning:** Bereinigung des Datensatzes von unvollständigen Fahrerprofilen ("Ghost-Riders") und irrelevanten Spalten.
* **Feature Engineering:** Erstellung neuer, aussagekräftiger Variablen wie das **Alter zum Zeitpunkt des Rennens** oder den **BMI**.
* **Imputation:** Umgang mit fehlenden Wetterdaten durch statistische Ersatzwerte (Median/Mittelwert).
* **Transformation:** Vorbereitung der Daten für den **XGBRanker** (Encoding kategorialer Variablen).

## 3. Grundsatz: GIGO (Garbage In - Garbage Out)
Gemäß den Vorlesungsunterlagen investieren wir ca. 80% der Projektzeit in diese Phase. Nur ein qualitativ hochwertiger und logisch konsistenter Datensatz ermöglicht es dem Modell später, präzise Vorhersagen zu treffen.

In [4]:
import pandas as pd
df = pd.read_pickle('../../data/processed/cleaned_master_data_with_coordinates_and_weather.pkl')


display(df.head(3))

print(df.columns.tolist())

# Liste der Spalten, die wir für das weitere Vorgehen und die ML-Modelle definitiv nicht brauchen
cols_to_drop = [
    'image_url',
    'place_of_birth'
]

df_cleaned = df.drop(columns=cols_to_drop)

,race,year,url,rank,rider_url,time_gap,birthdate,height,image_url,name,...,departure_wind_mittel,departure_luftfeuchte_mittel,departure_niederschlag_mittel,departure_windrichtung_mittel,arrival_temp_mittel,arrival_regen_mittel,arrival_wind_mittel,arrival_luftfeuchte_mittel,arrival_niederschlag_mittel,arrival_windrichtung_mittel
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,1,david-zabriskie,20:5120:51,1979-1-12,1.83,None,David Zabriskie,...,12.0,75.25,0.0,214.25,20.62,0.0,11.55,74.0,0.0,218.25
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,2,lance-armstrong,0:020:02,1971-9-18,1.78,None,Lance Armstrong,...,12.0,75.25,0.0,214.25,20.62,0.0,11.55,74.0,0.0,218.25
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,3,alexandr-vinokourov,0:530:53,1973-9-16,1.77,None,Alexandr Vinokurov,...,12.0,75.25,0.0,214.25,20.62,0.0,11.55,74.0,0.0,218.25


['race', 'year', 'url', 'rank', 'rider_url', 'time_gap', 'birthdate', 'height', 'image_url', 'name', 'nationality', 'place_of_birth', 'points_per_season_history', 'season_results', 'teams_history', 'weight', 'url_name', 'grand_tour_history', 'departure', 'arrival', 'distance', 'vertical_meters', 'profile_score', 'won_how', 'avg_speed', 'race_ranking', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 0, 'stage_nr', 'date', 'departure_lat', 'departure_lon', 'arrival_lat', 'arrival_lon', 'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel', 'departure_luftfeuchte_mittel', 'departure_niederschlag_mittel', 'departure_windrichtung_mittel', 'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel', 'arrival_luftfeuchte_mittel', 'arrival_niederschlag_mittel', 'arrival_windrichtung_mittel']


In [9]:
print(df_cleaned.columns.tolist())

pd.set_option('display.max_colwidth', None)

# Wir haben immer noch genestete Daten im Datensatz (points_per_season_hoistory, season_results, team_history, grand_tour_history)
# Diese werden nun nacheinander in ein "flaches Format gebracht"
print(df_cleaned.head(3)["points_per_season_history"])

['race', 'year', 'url', 'rank', 'rider_url', 'time_gap', 'birthdate', 'height', 'name', 'nationality', 'points_per_season_history', 'season_results', 'teams_history', 'weight', 'url_name', 'grand_tour_history', 'departure', 'arrival', 'distance', 'vertical_meters', 'profile_score', 'won_how', 'avg_speed', 'race_ranking', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 0, 'stage_nr', 'date', 'departure_lat', 'departure_lon', 'arrival_lat', 'arrival_lon', 'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel', 'departure_luftfeuchte_mittel', 'departure_niederschlag_mittel', 'departure_windrichtung_mittel', 'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel', 'arrival_luftfeuchte_mittel', 'arrival_niederschlag_mittel', 'arrival_windrichtung_mittel']
0                                                                                             [{'season': 2013, 'points': 1, 'rank': 2540}, {'season': 2012, 'points': 267, 'rank': 223}, {

In [16]:
import numpy as np

def get_season_stats(data):
    # Das Jahr des aktuellen Rennens
    target_year = data['year']
    history = data['points_per_season_history']

    # Falls keine Daten da sind oder die Spalte kein Listenformat hat
    if not isinstance(history, list):
        return pd.Series([np.nan, np.nan])

    # Suche in der Liste nach dem passenden Jahr
    for entry in history:
        if entry.get('season') == target_year:
            # Falls ein Key fehlt, auch hier NaN nutzen
            p = entry.get('points', np.nan)
            r = entry.get('rank', np.nan)
            return pd.Series([p, r])

    # Falls das Jahr in der Historie nicht vorkommt
    return pd.Series([np.nan, np.nan])

# Anwendung
df_cleaned[['rider_points_season', 'rider_rank_season']] = df_cleaned.apply(get_season_stats, axis=1)

print(df_cleaned[['year', 'rider_url','rider_points_season', 'rider_rank_season']])


        year            rider_url  rider_points_season  rider_rank_season
0       2005      david-zabriskie                  NaN                NaN
1       2005      lance-armstrong                  NaN                NaN
2       2005  alexandr-vinokourov               1562.0                4.0
3       2005      george-hincapie                 50.0              856.0
4       2005       laszlo-bodrogi                337.0              165.0
...      ...                  ...                  ...                ...
225687  2025             jan-hirt                 25.0             1112.0
225688  2025       nadav-raisberg                 80.0              616.0
225689  2025         jake-stewart                277.0              232.0
225690  2025         ethan-vernon                413.0              152.0
225691  2025   matthew-riccitello                593.0               87.0

[225692 rows x 4 columns]


In [19]:
# 1. Zufällige Stichprobe von 5 Fahrern ziehen
sample_names = df_cleaned['name'].dropna().unique()
random_sample_names = np.random.choice(sample_names, 5, replace=False)

# 2. Daten für diese Fahrer filtern
sample_df = df_cleaned[df_cleaned['name'].isin(random_sample_names)]

# 3. Relevante Spalten für den Check auswählen
# Wir sortieren nach Name und Jahr, um die historische Entwicklung zu sehen
check_cols = ['name', 'year', 'race', 'rider_points_season', 'rider_rank_season']
sample_output = sample_df[check_cols].sort_values(by=['name', 'year'])

# 4. Ausgabe
print("Test-Check für 5 zufällige Fahrer:")
pd.set_option('display.max_rows', None)
print(sample_output)
pd.reset_option('display.max_rows')

Test-Check für 5 zufällige Fahrer:
                       name  year             race  rider_points_season  \
215629         Milan Menten  2023  vuelta-a-espana                653.0   
215706         Milan Menten  2023  vuelta-a-espana                653.0   
215970         Milan Menten  2023  vuelta-a-espana                653.0   
215999         Milan Menten  2023  vuelta-a-espana                653.0   
216333         Milan Menten  2023  vuelta-a-espana                653.0   
216454         Milan Menten  2023  vuelta-a-espana                653.0   
216527         Milan Menten  2023  vuelta-a-espana                653.0   
216832         Milan Menten  2023  vuelta-a-espana                653.0   
216970         Milan Menten  2023  vuelta-a-espana                653.0   
217162         Milan Menten  2023  vuelta-a-espana                653.0   
217309         Milan Menten  2023  vuelta-a-espana                653.0   
217335         Milan Menten  2023  vuelta-a-espana               

In [22]:
print(df.columns.tolist())
print(df_cleaned['points_per_season_history'])

#points_per_season_history aus dem Datensatz löschen

df_cleaned.drop(columns=['points_per_season_history'], inplace=True)
#print(df_cleaned['points_per_season_history'])


['race', 'year', 'url', 'rank', 'rider_url', 'time_gap', 'birthdate', 'height', 'name', 'nationality', 'points_per_season_history', 'season_results', 'teams_history', 'weight', 'url_name', 'grand_tour_history', 'departure', 'arrival', 'distance', 'vertical_meters', 'profile_score', 'won_how', 'avg_speed', 'race_ranking', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 0, 'stage_nr', 'date', 'departure_lat', 'departure_lon', 'arrival_lat', 'arrival_lon', 'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel', 'departure_luftfeuchte_mittel', 'departure_niederschlag_mittel', 'departure_windrichtung_mittel', 'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel', 'arrival_luftfeuchte_mittel', 'arrival_niederschlag_mittel', 'arrival_windrichtung_mittel', 'rider_points_season', 'rider_rank_season']
0                                                                                                                                              

KeyError: 'points_per_season_history'

In [26]:
print(df_cleaned.columns.tolist())

# Als nächstes nehmen wir uns season_results vor
pd.set_option('display.max_colwidth', None)

print(df_cleaned.head(3)["season_results"])
# Da die Ergenisse über das Etappen Ranking bereits kommen, können wir die Spalte "droppen"
#Vermeidung Redundanzen, Data Leakage
df_cleaned.drop(columns=['season_results'], inplace=True)


['race', 'year', 'url', 'rank', 'rider_url', 'time_gap', 'birthdate', 'height', 'name', 'nationality', 'season_results', 'teams_history', 'weight', 'url_name', 'grand_tour_history', 'departure', 'arrival', 'distance', 'vertical_meters', 'profile_score', 'won_how', 'avg_speed', 'race_ranking', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 0, 'stage_nr', 'date', 'departure_lat', 'departure_lon', 'arrival_lat', 'arrival_lon', 'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel', 'departure_luftfeuchte_mittel', 'departure_niederschlag_mittel', 'departure_windrichtung_mittel', 'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel', 'arrival_luftfeuchte_mittel', 'arrival_niederschlag_mittel', 'arrival_windrichtung_mittel', 'rider_points_season', 'rider_rank_season']
0                                                                                                                                                                           

In [ ]:
print(df_cleaned.columns.tolist())

# Als nächstes nehmen wir uns season_results vor
pd.set_option('display.max_colwidth', None)

print(df_cleaned.head(3)["teams_history"])

# Team History muss wieder extrahiert werden

def get_team_stats(data):
    target_year = data['year']
    history = data['teams_history']

    if not isinstance(history, list):
        return pd.Series([np.nan, np.nan])

    for entry in history:
        if entry.get('season') == target_year:
            return pd.Series([entry.get('team_name', np.nan), entry.get('class', np.nan)])

    return pd.Series([np.nan, np.nan])

# Transformation anwenden
df_cleaned[['current_team', 'team_class']] = df_cleaned.apply(get_team_stats, axis=1)


# 1. Zufällige Stichprobe von 5 Fahrern ziehen
sample_names = df_cleaned['name'].dropna().unique()
random_five = np.random.choice(sample_names, 5, replace=False)

# 2. Daten für diese Fahrer filtern
test_sample = df_cleaned[df_cleaned['name'].isin(random_five)]

# 3. Wir sortieren nach Name und Jahr, um den Teamwechsel über die Zeit zu sehen
# Wir nehmen 'teams_history' mit rein, um es manuell mit 'current_team' zu vergleichen
check_cols = ['name', 'year', 'race', 'current_team', 'team_class', 'teams_history']
result = test_sample[check_cols].sort_values(by=['name', 'year'])

print("Erfolgskontrolle: Team-Mapping für 5 zufällige Fahrer")
pd.set_option('display.max_rows', None)
print(result)
pd.reset_option('display.max_rows')

In [ ]:
df_cleaned.drop(columns=['teams_history'], inplace=True)
#grand_tour_history können wir auch entfernen, da das nur interessant für das Scrapen der Fahrerprofile war
df_cleaned.drop(columns=['grand_tour_history'], inplace=True)

In [37]:
print(df_cleaned.columns.tolist())

# Überprüfen, ob noch genestete Daten in einer Spalte vorhanden sind:

# Zeige alle Spalten und ihre Datentypen
print(df_cleaned.dtypes)
# Die erste Zeile transponiert ausgeben, damit man alle Werte untereinander sieht

print(df_cleaned.iloc[0])

['race', 'year', 'url', 'rank', 'rider_url', 'time_gap', 'birthdate', 'height', 'name', 'nationality', 'weight', 'url_name', 'departure', 'arrival', 'distance', 'vertical_meters', 'profile_score', 'won_how', 'avg_speed', 'race_ranking', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 0, 'stage_nr', 'date', 'departure_lat', 'departure_lon', 'arrival_lat', 'arrival_lon', 'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel', 'departure_luftfeuchte_mittel', 'departure_niederschlag_mittel', 'departure_windrichtung_mittel', 'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel', 'arrival_luftfeuchte_mittel', 'arrival_niederschlag_mittel', 'arrival_windrichtung_mittel', 'rider_points_season', 'rider_rank_season', 'current_team', 'team_class']
race                                     object
year                                      int64
url                                      object
rank                                     object
rider_u

In [39]:
# Erstellung einer neuen .pkl Datei zur Zwischenspeicherung
import os

# Verzeichnis erstellen, falls es nicht existiert
output_dir = '../../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Dateiname mit aussagekräftigem Status (v2 für die flache Struktur)
file_name = '04_cleaned_master_data.pkl'
file_path = os.path.join(output_dir, file_name)

# Speichern
df_cleaned.to_pickle(file_path)


print(f"Datei gespeichert unter: {file_path}")
print(f"Aktuelle Shape: {df_cleaned.shape[0]} Zeilen, {df_cleaned.shape[1]} Spalten")


Datei gespeichert unter: ../../data/processed\04_cleaned_master_data.pkl
Aktuelle Shape: 225692 Zeilen, 49 Spalten


### 1. Flattening der Rider-Daten
Die Spalte `rider_results` enthielt Listen von Dictionaries. 
- Wir haben diese Listen "explodiert" (unnested).
- Für jeden Fahrer wurden die spezifischen UCI-Punkte der aktuellen Saison extrahiert.
- Ergebnis: Jede Zeile repräsentiert nun eindeutig eine Fahrer-Rennen-Kombination.

### 2. Team-Mapping & Historisierung
Da Fahrer jedes Jahr das Team wechseln können, haben wir eine Logik implementiert, die:
- Das korrekte Team des Fahrers basierend auf dem **Rennjahr** zuordnet.
- Den Team-Namen und die Team-Klasse (`current_team`, `team_class`) als flache Features bereitstellt.


### 4. Checkpointing
- **Speicherung:** Da der Preprocessing-Prozess rechenintensiv ist, wurde ein Checkpoint als Pickle-Datei erstellt.

## Wichtige Hinweise 

### Dateistruktur & Git
- **Pickle-Datei:** `data/processed/04_cleaned_master_data.pkl`
- **Git-Policy:** Die `.pkl` Datei ist in der `.gitignore` und wird **nicht** gepusht (Dateigröße & Best Practice).
- **Austausch:** Ich lade die aktuelle Datei in unsere Cloud (Drive) hoch. Bitte ladet sie euch lokal in euren `data/processed/` Ordner.






## **Strukturprüfung:** Identifikation von Datentypen und Skalenniveaus der Features.

In [40]:
df = pd.read_pickle('../../data/processed/04_cleaned_master_data.pkl')

In [42]:
# Ausgeben aller Datentypen

print(df.dtypes)
# noch sehr viel object Datentypen also nicht spezifiziert

race                                     object
year                                      int64
url                                      object
rank                                     object
rider_url                                object
time_gap                                 object
birthdate                                object
height                                  float64
name                                     object
nationality                              object
weight                                  float64
url_name                                 object
departure                                object
arrival                                  object
distance                                 object
vertical_meters                          object
profile_score                            object
won_how                                  object
avg_speed                                object
race_ranking                             object
one_day_races                           

In [43]:
# 1. Liste der Spalten, die eigentlich NUMERISCH sein müssen
to_numeric_cols = [
    'rank', 'distance', 'vertical_meters', 'profile_score', 'avg_speed',
    'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel',
    'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel'
]

for col in to_numeric_cols:
    if col in df.columns:
        # Alles entfernen, was keine Zahl oder Punkt ist (z.B. " km", " m", " °C")
        df[col] = df[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)
        # In Zahl umwandeln, Fehler (wie leere Strings) werden zu NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
print(df.dtypes)

#noch viele Float Werte aufgrund der NaNs in den Spalten

race                                     object
year                                      int64
url                                      object
rank                                    float64
rider_url                                object
time_gap                                 object
birthdate                                object
height                                  float64
name                                     object
nationality                              object
weight                                  float64
url_name                                 object
departure                                object
arrival                                  object
distance                                float64
vertical_meters                         float64
profile_score                           float64
won_how                                  object
avg_speed                               float64
race_ranking                             object
one_day_races                           